# ⚠️ Project 6.2 — Fault Code BST
**DSA for Mechatronics · Week 06 — Trees & BSTs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A PLC stores diagnostic fault codes in a **Binary Search Tree** so that:
- New faults are **inserted** in O(log n) average time
- The control room can **search** for a specific code instantly
- Resolved faults are **deleted** without rebuilding the structure
- The kth-most-critical fault can be found via **inorder traversal**

The BST ordering key is the fault code number.
Each node also stores a severity label and a description.


## Step 1 — Define the BST and insert your fault codes

In [ ]:
# ── Personalised parameters ────────────────────────────────────────
N_CODES       = random.randint(14, 22)
N_DELETE      = random.randint(2, 5)
K_SMALLEST    = random.randint(2, 6)
CODE_RANGE    = (random.randint(1000, 2000), random.randint(3000, 5000))

SEVERITIES    = ["INFO", "WARN", "ERROR", "CRIT"]
DESCRIPTIONS  = ["Overvoltage","Undervoltage","Overcurrent","Overtemp",
                 "CommLoss","SensorFail","ActuatorFail","MemFault",
                 "ConfigErr","CalibErr","PowerFail","BusTimeout",
                 "EncoderErr","MotorFault","EmergencyStop","SafetyViolation"]

fault_codes = sorted(random.sample(range(*CODE_RANGE), N_CODES))
# shuffle for insertion (BST shape depends on insertion order!)
insertion_order = fault_codes[:]
random.shuffle(insertion_order)

class BSTNode:
    def __init__(self, code, severity, desc):
        self.code     = code
        self.severity = severity
        self.desc     = desc
        self.left     = None
        self.right    = None

class FaultBST:
    def __init__(self):
        self.root = None
        self._size = 0

    def insert(self, code, severity, desc):
        """Insert a new fault code maintaining BST ordering. O(h)."""
        def _insert(node, code, severity, desc):
            if not node:
                return BSTNode(code, severity, desc)
            if code < node.code:
                node.left  = _insert(node.left,  code, severity, desc)
            elif code > node.code:
                node.right = _insert(node.right, code, severity, desc)
            # duplicate codes ignored
            return node
        self.root = _insert(self.root, code, severity, desc)
        self._size += 1

    def search(self, code):
        """Return node if found, else None. O(h)."""
        node = self.root
        while node:
            if code == node.code: return node
            node = node.left if code < node.code else node.right
        return None

    def inorder(self):
        """Return all nodes in ascending code order. O(n)."""
        result = []
        def _inorder(node):
            if not node: return
            _inorder(node.left)
            result.append(node)
            _inorder(node.right)
        _inorder(self.root)
        return result

    def kth_smallest(self, k):
        """Find kth smallest code (1-indexed) via inorder. O(n)."""
        nodes = self.inorder()
        return nodes[k-1] if 1 <= k <= len(nodes) else None

    def find_min(self):
        n = self.root
        while n and n.left: n = n.left
        return n

    def find_max(self):
        n = self.root
        while n and n.right: n = n.right
        return n

    def height(self):
        def _h(n): return 0 if not n else 1 + max(_h(n.left), _h(n.right))
        return _h(self.root)

bst = FaultBST()
for code in insertion_order:
    sev  = random.choice(SEVERITIES)
    desc = random.choice(DESCRIPTIONS)
    bst.insert(code, sev, desc)

sorted_nodes = bst.inorder()

print(f"Fault BST parameters:")
print(f"  Fault codes inserted : {N_CODES}")
print(f"  Code range           : {CODE_RANGE[0]} – {CODE_RANGE[1]}")
print(f"  BST height           : {bst.height()}")
print(f"  Insertion order      : {insertion_order[:8]}...")
print()
print(f"Inorder (sorted) fault list:")
print(f"  {'#':>3}  {'Code':>6}  {'Severity':<8}  Description")
print(f"  {'─'*3}  {'─'*6}  {'─'*8}  {'─'*18}")
for i, n in enumerate(sorted_nodes):
    print(f"  {i+1:3d}  {n.code:6d}  {n.severity:<8}  {n.desc}")


## Step 2 — Search and kth smallest

In [ ]:
# Search for a few specific codes (some present, some not)
search_targets = random.sample(fault_codes, 3) +                  random.sample([c for c in range(*CODE_RANGE) if c not in fault_codes], 2)
random.shuffle(search_targets)

print("Search results:")
print(f"  {'Code':>6}  {'Found':>6}  {'Severity':<8}  Description")
print(f"  {'─'*6}  {'─'*6}  {'─'*8}  {'─'*18}")
for code in search_targets:
    node = bst.search(code)
    if node:
        print(f"  {code:6d}  {'YES':>6}  {node.severity:<8}  {node.desc}")
    else:
        print(f"  {code:6d}  {'NO':>6}  {'—':<8}  —")

# kth smallest
kth_node = bst.kth_smallest(K_SMALLEST)
min_node = bst.find_min()
max_node = bst.find_max()

print()
print(f"BST extremes and rank queries:")
print(f"  Minimum code    : {min_node.code}  ({min_node.severity}: {min_node.desc})")
print(f"  Maximum code    : {max_node.code}  ({max_node.severity}: {max_node.desc})")
print(f"  {K_SMALLEST}th smallest code: {kth_node.code}  ({kth_node.severity}: {kth_node.desc})")


## Step 3 — Delete resolved faults (3-case BST deletion)

In [ ]:
def delete_bst(root, code):
    """
    Delete a node from BST. Three cases:
      1. Node has no children    → simply remove it.
      2. Node has one child      → replace with that child.
      3. Node has two children   → replace value with inorder successor
                                   (leftmost node of right subtree), then delete successor.
    """
    if not root:
        return root
    if code < root.code:
        root.left  = delete_bst(root.left,  code)
    elif code > root.code:
        root.right = delete_bst(root.right, code)
    else:
        # Found the node to delete
        if not root.left:                 # case 1 & 2: no left child
            return root.right
        if not root.right:                # case 2: no right child
            return root.left
        # case 3: two children — find inorder successor
        successor = root.right
        while successor.left:
            successor = successor.left
        root.code     = successor.code     # copy successor value up
        root.severity = successor.severity
        root.desc     = successor.desc
        root.right    = delete_bst(root.right, successor.code)
    return root

codes_to_delete = random.sample(fault_codes, N_DELETE)
print(f"Deleting {N_DELETE} resolved fault codes: {codes_to_delete}")
print()

for code in codes_to_delete:
    bst.root = delete_bst(bst.root, code)
    bst._size -= 1

remaining = bst.inorder()

print(f"After deletion ({len(remaining)} codes remain):")
print(f"  {'#':>3}  {'Code':>6}  {'Severity':<8}  Description")
print(f"  {'─'*3}  {'─'*6}  {'─'*8}  {'─'*18}")
for i, n in enumerate(remaining):
    print(f"  {i+1:3d}  {n.code:6d}  {n.severity:<8}  {n.desc}")

print(f"\n  New BST height : {bst.height()}")


## Step 4 — Balance check and severity summary

In [ ]:
def is_balanced(root):
    """
    Check if BST is height-balanced: for every node,
    |height(left) - height(right)| <= 1.
    Returns (is_balanced, height). Returns (-1, -1) if unbalanced (early exit).
    """
    def check(node):
        if not node: return (True, 0)
        lb, lh = check(node.left)
        if not lb: return (False, -1)
        rb, rh = check(node.right)
        if not rb: return (False, -1)
        balanced = abs(lh - rh) <= 1
        return (balanced, 1 + max(lh, rh))
    return check(root)

from collections import Counter
balanced, final_height = is_balanced(bst.root)
sev_counts = Counter(n.severity for n in remaining)

print(f"BST balance check:")
print(f"  Balanced : {'YES ✅' if balanced else 'NO ⚠ (may degrade to O(n))'}")
print(f"  Height   : {final_height}")
print(f"  Nodes    : {len(remaining)}")
print()
print("Severity breakdown (remaining faults):")
for sev in SEVERITIES:
    count = sev_counts.get(sev, 0)
    bar   = "█" * count
    print(f"  {sev:<6}: {bar} ({count})")

crit_count = sev_counts.get("CRIT", 0)
status = "🔴 CRITICAL FAULTS ACTIVE" if crit_count > 0 else "🟡 NO CRITICAL FAULTS"
print(f"\n  System status: {status}")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " FAULT CODE BST — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  BST PARAMETERS" + " "*(W-16) + "║")
print(f"║  {'Codes inserted':<26}: {N_CODES:<26}║")
print(f"║  {'Code range':<26}: {CODE_RANGE[0]} – {CODE_RANGE[1]}{'':<18}║")
print(f"║  {'Initial height':<26}: {bst.height() + (1 if N_DELETE > 0 else 0):<26}║")
print("╠" + "═"*W + "╣")
print("║  SEARCH & RANK" + " "*(W-15) + "║")
print(f"║  {'Minimum fault code':<26}: {min_node.code}  ({min_node.severity}){'':<14}║")
print(f"║  {'Maximum fault code':<26}: {max_node.code}  ({max_node.severity}){'':<14}║")
print(f"║  {str(K_SMALLEST)+'th smallest code':<26}: {kth_node.code}  ({kth_node.severity}){'':<14}║")
print("╠" + "═"*W + "╣")
print("║  AFTER DELETION" + " "*(W-16) + "║")
print(f"║  {'Codes deleted':<26}: {N_DELETE:<26}║")
print(f"║  {'Codes remaining':<26}: {len(remaining):<26}║")
print(f"║  {'Height after deletion':<26}: {final_height:<26}║")
print(f"║  {'BST balanced':<26}: {'YES' if balanced else 'NO':<26}║")
print("╠" + "═"*W + "╣")
print("║  SEVERITY SUMMARY" + " "*(W-18) + "║")
for sev in SEVERITIES:
    print(f"║  {sev:<26}: {sev_counts.get(sev,0):<26}║")
print(f"║  {'System status':<26}: {status[:26]:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which tree concept did you find most important, and why?**
Pick one technique from the notebook (traversal, BST property, recursion pattern…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
